# LAB02 — Auditoría del dataset aprobado

## Dataset seleccionado

**Disease Prediction Using Machine Learning**

- Fuente: Kaggle
- Target: `prognosis`
- Tipo de problema: clasificación multiclase
- Unidad de análisis: registro de síntomas asociado a una categoría de enfermedad
- Uso: académico

## Objetivo del notebook

Auditar la estructura, calidad y consistencia del dataset antes de construir los modelos de clasificación. Se revisarán dimensiones, tipos de datos, valores ausentes, duplicados, distribución del target y posibles variables que deban excluirse para evitar ruido o fuga de información.

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
current_dir = Path.cwd()

if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

data_path = (
    project_root
    / "data"
    / "raw"
    / "disease_prediction_source"
    / "Training.csv"
)

df = pd.read_csv(data_path)

print("Ruta:", data_path)
print("Dimensiones originales:", df.shape)

Ruta: c:\Users\Gladys\INF8239_U01\data\raw\disease_prediction_source\Training.csv
Dimensiones originales: (4920, 134)


## Revisión de columna residual

Durante la inspección inicial se identificó la columna `Unnamed: 133`, presente únicamente en el archivo de entrenamiento. Antes de excluirla, se verificará si contiene información útil o si corresponde a una columna residual generada durante la exportación del CSV.


In [3]:
extra_column = "Unnamed: 133"

print("Columna presente:", extra_column in df.columns)
print("Valores ausentes:", df[extra_column].isna().sum(), "de", len(df))
print("Valores no nulos:", df[extra_column].notna().sum())

assert df[extra_column].isna().all(), (
    f"La columna {extra_column} contiene valores y no debe eliminarse automáticamente."
)

df_clean = df.drop(columns=[extra_column]).copy()

print("Dimensiones después de excluir la columna residual:", df_clean.shape)

Columna presente: True
Valores ausentes: 4920 de 4920
Valores no nulos: 0
Dimensiones después de excluir la columna residual: (4920, 133)


## Auditoría de calidad del dataset

Después de excluir la columna residual vacía, se evaluarán los tipos de datos, los valores ausentes, los registros duplicados y la distribución de la variable objetivo `prognosis`. Esta revisión permite identificar problemas de calidad antes de construir los modelos de clasificación.

In [4]:
print("Dimensiones del dataset limpio:", df_clean.shape)

print("\nTipos de datos:")
print(df_clean.dtypes.value_counts())

print("\nValores ausentes totales:")
print(df_clean.isna().sum().sum())

print("\nFilas duplicadas:")
print(df_clean.duplicated().sum())

print("\nNúmero de clases del target:")
print(df_clean["prognosis"].nunique())

print("\nDistribución de clases:")
print(df_clean["prognosis"].value_counts().sort_index())

Dimensiones del dataset limpio: (4920, 133)

Tipos de datos:
int64    132
str        1
Name: count, dtype: int64

Valores ausentes totales:
0

Filas duplicadas:
4616

Número de clases del target:
41

Distribución de clases:
prognosis
(vertigo) Paroymsal  Positional Vertigo    120
AIDS                                       120
Acne                                       120
Alcoholic hepatitis                        120
Allergy                                    120
Arthritis                                  120
Bronchial Asthma                           120
Cervical spondylosis                       120
Chicken pox                                120
Chronic cholestasis                        120
Common Cold                                120
Dengue                                     120
Diabetes                                   120
Dimorphic hemmorhoids(piles)               120
Drug Reaction                              120
Fungal infection                           120
GERD          

## Revisión de registros duplicados

La auditoría identificó una cantidad elevada de registros duplicados exactos. Debido a que una partición aleatoria podría colocar registros idénticos tanto en entrenamiento como en prueba, se analizará la estructura de los duplicados antes de decidir su tratamiento.

La eliminación automática de duplicados tampoco se realizará sin evaluación previa, ya que podría reducir sustancialmente el número efectivo de observaciones del dataset.

In [5]:
total_rows = len(df_clean)
duplicate_rows = df_clean.duplicated().sum()
unique_rows = len(df_clean.drop_duplicates())

print("Total de registros:", total_rows)
print("Registros duplicados:", duplicate_rows)
print("Registros únicos:", unique_rows)

unique_by_class = (
    df_clean
    .drop_duplicates()
    ["prognosis"]
    .value_counts()
    .sort_index()
)

print("\nRegistros únicos por clase:")
print(unique_by_class)

print("\nResumen de registros únicos por clase:")
print(unique_by_class.describe())

Total de registros: 4920
Registros duplicados: 4616
Registros únicos: 304

Registros únicos por clase:
prognosis
(vertigo) Paroymsal  Positional Vertigo     7
AIDS                                        5
Acne                                        5
Alcoholic hepatitis                         8
Allergy                                     5
Arthritis                                   6
Bronchial Asthma                            7
Cervical spondylosis                        6
Chicken pox                                10
Chronic cholestasis                         8
Common Cold                                 9
Dengue                                     10
Diabetes                                    9
Dimorphic hemmorhoids(piles)                6
Drug Reaction                               6
Fungal infection                            5
GERD                                        7
Gastroenteritis                             5
Heart attack                                5
Hepatitis B  

## Conclusión de la auditoría preliminar

El dataset contiene 4,920 registros después de excluir la columna residual vacía; sin embargo, se identificaron 4,616 registros duplicados exactos, por lo que solamente existen 304 observaciones únicas.

La elevada repetición implica un riesgo significativo de contaminación entre los conjuntos de entrenamiento y prueba si se utiliza una partición aleatoria convencional, ya que un mismo patrón de síntomas podría aparecer en ambos conjuntos. Esto podría generar métricas de desempeño artificialmente elevadas.

Por otra parte, la eliminación de todos los duplicados reduciría el conjunto a 304 observaciones únicas, cantidad inferior al mínimo de 500 registros establecido como criterio general de aceptación para el ejercicio.

En consecuencia, este dataset no se considera adecuado como opción principal para continuar el experimento bajo el protocolo establecido. Se utilizará el segundo dataset previamente aprobado por el docente, Company Financials Dataset, sujeto a su correspondiente auditoría de calidad y riesgo de fuga de información.